# Deep Cerebellar Nucleis SDF and CR% (Reproduce Figure 3-5)

The figure reports the SDF in DCN population (averaging across cell SDFs) for each trial. The progress of the trials is color-mapped by the horizontal color bar below. The vertical lines represent the CS-onset (light blue), US-onset (light red), and CS-US co-termination (lilac). In grey are shown the two windows, baseline and CR window, for evaluation of functional plasticity.

### DCN-CS = Population of PC receiving at least 3 receiving PC conveying CS stimuli
### DNC-noise = Population of PC receiving less than 3 of receiving PC conveying CS stimuli

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils import get_spike_activity, sdf, sdf_mean
import json
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import matplotlib.patches as patches
import scipy.stats as st
import h5py

n_sim = input("Number of simulations: ")
folder_path = "results"  # Set the folder name
thr_perc = 0.30
thr_pc_cs = 3
thr_pc_cs = 3

with open("/g100_work/EIRI_E_POLIM2/no_paper/NODS/network_configuration.json", "r") as json_file:
    net_config = json.load(json_file)
CS_burst_dur = net_config["devices"]["CS"]["parameters"]["burst_dur"]
CS_start_first = float(net_config["devices"]["CS"]["parameters"]["start_first"])
between_start = net_config["devices"]["CS"]["parameters"]["between_start"]
n_trials = net_config["devices"]["CS"]["parameters"]["n_trials"]
US_start_first = float(net_config["devices"]["US"]["parameters"]["start_first"])
CS_color = net_config["colors"]["CS"]
US_color = net_config["colors"]["US"]
cell_color = net_config["cell_types"]["purkinje_cell"]["color"][0]
with_NO_color = net_config["devices"]["nNOS"]["color"][0]

palette = list(reversed(sns.color_palette("viridis", n_trials).as_hex()))
sm = plt.cm.ScalarMappable(cmap="viridis_r", norm=plt.Normalize(vmin=0, vmax=n_trials))

In [ ]:
# Define the moving_average_left function that's missing in the original code
def moving_average_left(data, window):
    """
    Calculate the moving average with window size window.
    For each point, the average includes the current point and window-1 preceding points.
    """
    result = np.zeros_like(data, dtype=float)
    for i in range(len(data)):
        start = max(0, i - window + 1)
        result[i] = np.mean(data[start:i+1])
    return result

# Define the moving_average_left function that's missing in the original code
def moving_average_left(data, window):
    """
    Calculate the moving average with window size window.
    For each point, the average includes the current point and window-1 preceding points.
    """
    result = np.zeros_like(data, dtype=float)
    for i in range(len(data)):
        start = max(0, i - window + 1)
        result[i] = np.mean(data[start:i+1])
    return result

In [ ]:
with open("/g100_work/EIRI_E_POLIM2/no_paper/NODS/network_configuration.json", "r") as json_file:
    net_config = json.load(json_file)
CS_burst_dur = net_config["devices"]["CS"]["parameters"]["burst_dur"]
CS_start_first = float(net_config["devices"]["CS"]["parameters"]["start_first"])
between_start = net_config["devices"]["CS"]["parameters"]["between_start"]
n_trials = net_config["devices"]["CS"]["parameters"]["n_trials"]
US_start_first = float(net_config["devices"]["US"]["parameters"]["start_first"])
CS_color = net_config["colors"]["CS"]
US_color = net_config["colors"]["US"]
cell_color = net_config["cell_types"]["purkinje_cell"]["color"][0]
with_NO_color = net_config["devices"]["nNOS"]["color"][0]
without_NO_color = "#000000"
palette = list(reversed(sns.color_palette("viridis", n_trials).as_hex()))
sm = plt.cm.ScalarMappable(cmap="viridis_r", norm=plt.Normalize(vmin=0, vmax=n_trials))


file_path = '/g100_work/EIRI_E_POLIM2/no_paper/NODS/data/pfs-PC_CS_40.pkl'
with open(file_path, 'rb') as file:
    id_granule = pickle.load(file)
    file.close()

rel_dist_path = '/g100_work/EIRI_E_POLIM2/no_paper/NODS/data/relative_dist.csv'
# source_id, nos_id, ev_points_id, d, cluster
relative_dist = pd.read_csv(rel_dist_path, header = None)
relative_dist = relative_dist.iloc[:,1:]
relative_dist.columns = ['source_id', 'nos_id', 'ev_points_id', 'd', 'cluster']
matched_grc = relative_dist[relative_dist['source_id'].isin(id_granule)]

pc_ids = np.unique(relative_dist['cluster'])
total_synapses_per_pc = relative_dist.groupby('cluster').size()
matched_synapses_per_pc = matched_grc.groupby('cluster').size()
pc_ratio = (matched_synapses_per_pc / total_synapses_per_pc).fillna(0)
pc_ratio_df = pc_ratio.reset_index()
pc_ratio_df.columns = ['PC_cluster', 'matched_ratio']
pc_ratio_df['CS_syn'] = pc_ratio_df['matched_ratio'].apply(
    lambda x: 'CS' if x >= thr_perc else 'no_CS'
)

cs_pc_ids = pc_ratio_df[pc_ratio_df['CS_syn'] == 'CS']['PC_cluster'].values
no_cs_pc_ids = pc_ratio_df[pc_ratio_df['CS_syn'] == 'no_CS']['PC_cluster'].values

file_path = '/g100_work/EIRI_E_POLIM2/no_paper/NODS/data/PC_DCN.pkl'
with open(file_path, 'rb') as file:
    pc_dcn_syn = pickle.load(file)
    file.close()
pc_dcn_df = pd.DataFrame(pc_dcn_syn)
pc_dcn_df.columns = ['PC', 'DCN']
dcn_cs_ids = np.unique(pc_dcn_df[pc_dcn_df['PC'].isin(cs_pc_ids)]['DCN'].values)
dcn_no_cs_ids = np.unique(pc_dcn_df[pc_dcn_df['PC'].isin(no_cs_pc_ids)]['DCN'].values)
cs_conn_counts = pc_dcn_df[pc_dcn_df['PC'].isin(cs_pc_ids)].groupby('DCN').size()
mean_PC_DCN = np.mean(cs_conn_counts)
# Set default category to 'no_CS'
dcn_category = pd.Series('no_CS', index=pc_dcn_df['DCN'].unique())

# Update to 'CS' if DCN gets input from > number of CS PCs

dcn_category.loc[cs_conn_counts[cs_conn_counts >= thr_pc_cs].index] = 'CS'

# Create final DataFrame
dcn_df = dcn_category.reset_index()
dcn_df.columns = ['DCN', 'CS_input']
cs_dcn_ids = dcn_df[dcn_df['CS_input'] == 'CS']['DCN'].values
no_cs_dcn_ids = dcn_df[dcn_df['CS_input'] == 'no_CS']['DCN'].values

result_path = "/g100_scratch/userexternal/csartor1/results/Paper/"
cell = "dcn"
folder_path = result_path + f"without_NO/"
folder_path_NO = result_path + f"{NO_folder}/minus50_plus10/"

step = 5

t_bs = 175  # Baseline time index, adjust as needed

In [ ]:
for noise in [0, 4, 8]:

    # DCN Processing
    sdf_mean_trials_simulations_dcn = []
    sdf_mean_trials_simulations_dcn_NO = []
    
    sdf_mean_trials_simulations_cs_dcn = []
    sdf_mean_trials_simulations_no_cs_dcn = []
    
    sdf_mean_trials_simulations_cs_dcn_NO = []
    sdf_mean_trials_simulations_no_cs_dcn_NO = []
    

    for k in range(n_sim):
        # Process DCN data
        sdf_mean_dcn = []
        sdf_mean_dcn_NO = []
        
        file_path = folder_path + f'simulation_{noise}Hz_sim{k}'
        dcn_spk = get_spike_activity(cell_name=cell, path=file_path)
        sdf_mean_cs_dcn = []
        sdf_mean_no_cs_dcn = []
        dcn_spk_cs = dcn_spk[np.isin(dcn_spk[:, 0], cs_dcn_ids)]
        dcn_spk_no_cs = dcn_spk[np.isin(dcn_spk[:, 0], no_cs_dcn_ids)]

        file_path_NO = folder_path_NO + f'{noise}Hz/sim{k}'
        dcn_spk_NO = get_spike_activity(cell_name=cell, path=file_path_NO)
        sdf_mean_cs_dcn_NO = []
        sdf_mean_no_cs_dcn_NO = []
        dcn_spk_cs_NO = dcn_spk_NO[np.isin(dcn_spk_NO[:, 0], cs_dcn_ids)]
        dcn_spk_no_cs_NO = dcn_spk_NO[np.isin(dcn_spk_NO[:, 0], no_cs_dcn_ids)]

        for trial in range(n_trials):

            start = trial * between_start
            stop = CS_start_first + CS_burst_dur + trial * between_start

            sdf_dcn = sdf(start=start, stop=stop, spk=dcn_spk, step=step)
            sdf_mean_dcn.append(sdf_mean(sdf_dcn))

            sdf_dcn_NO = sdf(start=start, stop=stop, spk=dcn_spk_NO, step=step)
            sdf_mean_dcn_NO.append(sdf_mean(sdf_dcn_NO))
            
            # DCN - CS DCNs
            sdf_cs_dcn = sdf(start=start, stop=stop, spk=dcn_spk_cs, step=step)
            sdf_mean_cs_dcn.append(sdf_mean(sdf_cs_dcn))
        
            # DCN - no_CS DCNs
            sdf_no_cs_dcn = sdf(start=start, stop=stop, spk=dcn_spk_no_cs, step=step)
            sdf_mean_no_cs_dcn.append(sdf_mean(sdf_no_cs_dcn))

            # DCN - CS DCNs NO
            sdf_cs_dcn_NO = sdf(start=start, stop=stop, spk=dcn_spk_cs_NO, step=step)
            sdf_mean_cs_dcn_NO.append(sdf_mean(sdf_cs_dcn_NO))
        
            # DCN - no_CS DCNs NO
            sdf_no_cs_dcn_NO = sdf(start=start, stop=stop, spk=dcn_spk_no_cs_NO, step=step)
            sdf_mean_no_cs_dcn_NO.append(sdf_mean(sdf_no_cs_dcn_NO))

        # DCN data
        sdf_mean_over_trials = np.array(sdf_mean_dcn)
        sdf_mean_over_trials_NO = np.array(sdf_mean_dcn_NO)
        sdf_mean_trials_simulations_dcn.append(sdf_mean_over_trials)
        sdf_mean_trials_simulations_dcn_NO.append(sdf_mean_over_trials_NO)
        
        sdf_mean_over_trials_cs_dcn = np.array(sdf_mean_cs_dcn)
        sdf_mean_over_trials_no_cs_dcn = np.array(sdf_mean_no_cs_dcn)
        
        sdf_mean_trials_simulations_cs_dcn.append(sdf_mean_over_trials_cs_dcn)
        sdf_mean_trials_simulations_no_cs_dcn.append(sdf_mean_over_trials_no_cs_dcn)

        sdf_mean_over_trials_cs_dcn_NO = np.array(sdf_mean_cs_dcn_NO)
        sdf_mean_over_trials_no_cs_dcn_NO = np.array(sdf_mean_no_cs_dcn_NO)
        
        sdf_mean_trials_simulations_cs_dcn_NO.append(sdf_mean_over_trials_cs_dcn_NO)
        sdf_mean_trials_simulations_no_cs_dcn_NO.append(sdf_mean_over_trials_no_cs_dcn_NO)

    # DCN data processing
    stack_sdf_mean = np.stack(sdf_mean_trials_simulations_dcn, axis=0)
    stack_sdf_mean_NO = np.stack(sdf_mean_trials_simulations_dcn_NO, axis=0)
    
    stack_sdf_mean_cs_dcn = np.stack(sdf_mean_trials_simulations_cs_dcn, axis=0)
    stack_sdf_mean_no_cs_dcn = np.stack(sdf_mean_trials_simulations_no_cs_dcn, axis=0)
    
    stack_sdf_mean_cs_dcn_NO = np.stack(sdf_mean_trials_simulations_cs_dcn_NO, axis=0)
    stack_sdf_mean_no_cs_dcn_NO = np.stack(sdf_mean_trials_simulations_no_cs_dcn_NO, axis=0)
    
    # Calculate DCN baseline for normalization
    sdf_dcn_bs = []
    sdf_dcn_bs_NO = []
    
    sdf_cs_dcn_bs = []
    sdf_cs_dcn_bs_NO = []
    sdf_no_cs_dcn_bs = []
    sdf_no_cs_dcn_bs_NO = []
    
    for i in range(n_sim):
        for j in range(1, 4):
            sdf_dcn_bs.append(stack_sdf_mean[i, j, t_bs])
            sdf_dcn_bs_NO.append(stack_sdf_mean_NO[i, j, t_bs])
            

    sdf_dcn_bs_median = np.median(sdf_dcn_bs) if sdf_dcn_bs else 0
    sdf_dcn_bs_median_NO = np.median(sdf_dcn_bs_NO) if sdf_dcn_bs_NO else 0
    
    # Calculate median across simulations
    median_sdf_cs_dcn = np.median(stack_sdf_mean_cs_dcn, axis=0)
    median_sdf_no_cs_dcn = np.median(stack_sdf_mean_no_cs_dcn, axis=0)
    median_sdf_cs_dcn_NO = np.median(stack_sdf_mean_cs_dcn_NO, axis=0)
    median_sdf_no_cs_dcn_NO = np.median(stack_sdf_mean_no_cs_dcn_NO, axis=0)
    
    # Normalize by subtracting baseline
    sdf_norm_cs_dcn = median_sdf_cs_dcn - sdf_dcn_bs_median
    sdf_norm_cs_dcn_NO = median_sdf_cs_dcn_NO - sdf_dcn_bs_median_NO
    sdf_norm_no_cs_dcn = median_sdf_no_cs_dcn - sdf_dcn_bs_median
    sdf_norm_no_cs_dcn_NO = median_sdf_no_cs_dcn_NO - sdf_dcn_bs_median_NO

    palette = list(reversed(sns.color_palette("viridis", n_trials).as_hex()))
    sm = plt.cm.ScalarMappable(
        cmap="viridis_r", norm=plt.Normalize(vmin=0, vmax=n_trials)
    )

    # Plot for DCN CS > threshold
    plt.rcParams.update({'font.size': 16})
    fig_sdf, axs_sdf = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
    
    for trial in range(n_trials):
        axs_sdf[0].plot(sdf_norm_cs_dcn[trial], palette[trial])
        axs_sdf[1].plot(sdf_norm_cs_dcn_NO[trial], palette[trial])
    
    axs_sdf[0].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[0].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[0].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    axs_sdf[1].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[1].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[1].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    
    axs_sdf[0].axhline(
        thr, linewidth=2, c='black', linestyle='--', label="Threshold"
    )
    axs_sdf[1].axhline(
        thr, linewidth=2, c='black', linestyle='--', label="Threshold"
    )
    
    axs_sdf[0].set_ylabel("SDF [Hz]")
    axs_sdf[0].set_xlabel("Time [ms]")
    axs_sdf[1].set_xlabel("Time [ms]")
    #axs_sdf[0].legend()
    #axs_sdf[1].legend()
    cbar = plt.colorbar(sm, ax=axs_sdf.ravel().tolist(), orientation='horizontal')
    cbar.set_label('Trials')
    
    fig_sdf.suptitle(
        f"DCN-CS SDF with STDP vs with NO-STDP: {noise}Hz", fontsize=18
    , fontweight='bold')
    lines1, labels1 = axs_sdf[0].get_legend_handles_labels()
    fig_sdf.legend(lines1, labels1, loc='lower center', ncol=3, 
                   #bbox_to_anchor=(0.5, 0.98), 
                   frameon=True)
    plt.show()
    plt.close()
    
    # Plot for DCN CS < threshold
    plt.rcParams.update({'font.size': 16})
    fig_sdf, axs_sdf = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
    
    for trial in range(n_trials):
        axs_sdf[0].plot(sdf_norm_no_cs_dcn[trial], palette[trial])
        axs_sdf[1].plot(sdf_norm_no_cs_dcn_NO[trial], palette[trial])
    
    axs_sdf[0].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[0].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[0].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    axs_sdf[1].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[1].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[1].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    
    axs_sdf[0].axhline(
        thr, linewidth=2, c='black', linestyle='--', label="Threshold"
    )
    axs_sdf[1].axhline(
        thr, linewidth=2, c='black', linestyle='--', label="Threshold"
    )
    
    axs_sdf[0].set_ylabel("SDF [Hz]")
    axs_sdf[0].set_xlabel("Time [ms]")
    axs_sdf[1].set_xlabel("Time [ms]")

    cbar = plt.colorbar(sm, ax=axs_sdf.ravel().tolist(), orientation='horizontal')
    cbar.set_label('Trials')
    
    fig_sdf.suptitle(
        f"DCN-noise SDF with STDP vs with NO-STDP: {noise}Hz", fontsize=16
    , fontweight='bold')
    lines1, labels1 = axs_sdf[0].get_legend_handles_labels()
    fig_sdf.legend(lines1, labels1, loc='lower center', ncol=3, 
                   #bbox_to_anchor=(0.5, 0.98), 
                   frameon=True)
    plt.show()
    plt.close()
    
    # CR% analysis for CS PCs
    cr_perc_sim_cs = []
    bs_sim_cs = []
    sdf_mean_sim_arr_cs = np.array(stack_sdf_mean_cs_dcn)
    
    cr_perc_sim_cs_NO = []
    bs_sim_cs_NO = []
    sdf_mean_sim_arr_cs_NO = np.array(stack_sdf_mean_cs_dcn_NO)
    
    for sim in range(n_sim):
        bs_sim_cs = np.median(sdf_mean_sim_arr_cs[sim, 1:4, t_bs])
        cr_bs_cs = sdf_mean_sim_arr_cs[sim] - bs_sim_cs
        over_thr_cs = []
        
        bs_sim_cs_NO = np.median(sdf_mean_sim_arr_cs_NO[sim, 1:4, t_bs])
        cr_bs_cs_NO = sdf_mean_sim_arr_cs_NO[sim] - bs_sim_cs_NO
        over_thr_cs_NO = []
        
        for trial in range(n_trials):
            sdf_over_cs = cr_bs_cs[trial] > thr
            
            sdf_over_bs_cs = len(np.where(sdf_over_cs[149:200])[0]) / (len(range(149, 200)))
            sdf_over_cr_cs = len(np.where(sdf_over_cs[215:300])[0]) / (len(range(215, 300)))
            
            if sdf_over_bs_cs < 0.75 and sdf_over_cr_cs >= 0.75:
                over_thr_cs.append(1)
            else:
                over_thr_cs.append(0)
                
            sdf_over_cs_NO = cr_bs_cs_NO[trial] > thr
            
            sdf_over_bs_cs_NO = len(np.where(sdf_over_cs_NO[149:200])[0]) / (len(range(149, 200)))
            sdf_over_cr_cs_NO = len(np.where(sdf_over_cs_NO[215:300])[0]) / (len(range(215, 300)))
            
            if sdf_over_bs_cs_NO < 0.75 and sdf_over_cr_cs_NO >= 0.75:
                over_thr_cs_NO.append(1)
            else:
                over_thr_cs_NO.append(0)
                
        cr_perc_avg_cs = moving_average_left(over_thr_cs, 10) * 100
        cr_perc_sim_cs.append(cr_perc_avg_cs)
        cr_perc_avg_cs_NO = moving_average_left(over_thr_cs_NO, 10) * 100
        cr_perc_sim_cs_NO.append(cr_perc_avg_cs_NO)
        
    cr_perc_sim_cs = np.stack(cr_perc_sim_cs, axis=0)
    cr_perc_sim_cs_NO = np.stack(cr_perc_sim_cs_NO, axis=0)
    
    median_cr_cs = np.median(cr_perc_sim_cs, axis=0)
    q1_cr_cs = np.percentile(cr_perc_sim_cs, 25, axis=0)
    q3_cr_cs = np.percentile(cr_perc_sim_cs, 75, axis=0)
    
    median_cr_cs_NO = np.median(cr_perc_sim_cs_NO, axis=0)
    q1_cr_cs_NO = np.percentile(cr_perc_sim_cs_NO, 25, axis=0)
    q3_cr_cs_NO = np.percentile(cr_perc_sim_cs_NO, 75, axis=0)
    
    # CR% analysis for no_CS PCs
    cr_perc_sim_no_cs = []
    bs_sim_no_cs = []
    sdf_mean_sim_arr_no_cs = np.array(stack_sdf_mean_no_cs_dcn)
    
    cr_perc_sim_no_cs_NO = []
    bs_sim_no_cs_NO = []
    sdf_mean_sim_arr_no_cs_NO = np.array(stack_sdf_mean_no_cs_dcn_NO)
    
    for sim in range(n_sim):
        bs_sim_no_cs = np.median(sdf_mean_sim_arr_no_cs[sim, 1:4, t_bs])
        cr_bs_no_cs = sdf_mean_sim_arr_no_cs[sim] - bs_sim_no_cs
        over_thr_no_cs = []
        
        bs_sim_no_cs_NO = np.median(sdf_mean_sim_arr_no_cs_NO[sim, 1:4, t_bs])
        cr_bs_no_cs_NO = sdf_mean_sim_arr_no_cs_NO[sim] - bs_sim_no_cs_NO
        over_thr_no_cs_NO = []
        
        for trial in range(n_trials):
            sdf_over_no_cs = cr_bs_no_cs[trial] > thr
            
            sdf_over_bs_no_cs = len(np.where(sdf_over_no_cs[149:t_bs])[0]) / (len(range(149, t_bs)))
            sdf_over_cr_no_cs = len(np.where(sdf_over_no_cs[215:300])[0]) / (len(range(215, 300)))
            
            if sdf_over_bs_no_cs < 0.75 and sdf_over_cr_no_cs >= 0.75:
                over_thr_no_cs.append(1)
            else:
                over_thr_no_cs.append(0)
                
            sdf_over_no_cs_NO = cr_bs_no_cs_NO[trial] > thr
            
            sdf_over_bs_no_cs_NO = len(np.where(sdf_over_no_cs_NO[149:t_bs])[0]) / (len(range(149, t_bs)))
            sdf_over_cr_no_cs_NO = len(np.where(sdf_over_no_cs_NO[215:300])[0]) / (len(range(215, 300)))
            
            if sdf_over_bs_no_cs_NO < 0.75 and sdf_over_cr_no_cs_NO >= 0.75:
                over_thr_no_cs_NO.append(1)
            else:
                over_thr_no_cs_NO.append(0)
                
        cr_perc_avg_no_cs = moving_average_left(over_thr_no_cs, 10) * 100
        cr_perc_sim_no_cs.append(cr_perc_avg_no_cs)
        cr_perc_avg_no_cs_NO = moving_average_left(over_thr_no_cs_NO, 10) * 100
        cr_perc_sim_no_cs_NO.append(cr_perc_avg_no_cs_NO)
        
    cr_perc_sim_no_cs = np.stack(cr_perc_sim_no_cs, axis=0)
    cr_perc_sim_no_cs_NO = np.stack(cr_perc_sim_no_cs_NO, axis=0)
    
    median_cr_no_cs = np.median(cr_perc_sim_no_cs, axis=0)
    q1_cr_no_cs = np.percentile(cr_perc_sim_no_cs, 25, axis=0)
    q3_cr_no_cs = np.percentile(cr_perc_sim_no_cs, 75, axis=0)
    
    median_cr_no_cs_NO = np.median(cr_perc_sim_no_cs_NO, axis=0)
    q1_cr_no_cs_NO = np.percentile(cr_perc_sim_no_cs_NO, 25, axis=0)
    q3_cr_no_cs_NO = np.percentile(cr_perc_sim_no_cs_NO, 75, axis=0)
    
    # Create the first plot: CS PCs with and without NO
    plt.figure(figsize=(8, 8))
    plt.rcParams.update({'font.size': 16})
    x = range(1, n_trials+1)
    
    plt.errorbar(x, median_cr_cs, 
                 yerr=[median_cr_cs - q1_cr_cs, q3_cr_cs - median_cr_cs],
                 fmt='o-', 
                 label=f"without NO", 
                 capsize=5, 
                 markersize=3,
                 ecolor='black',
                 color='black')
    plt.errorbar(x, median_cr_cs_NO, 
                 yerr=[median_cr_cs_NO - q1_cr_cs_NO, q3_cr_cs_NO - median_cr_cs_NO],
                 fmt='o-', 
                 label=f"with NO", 
                 capsize=5, 
                 markersize=3,
                 ecolor=with_NO_color,
                 color=with_NO_color)
    
    plt.xlabel('Trial set [#]')
    plt.ylabel('CR [%]')
    plt.xticks([1, 5, 10, 15, 20, 25, 30])
    plt.title(f'Conditioned Response: DCN-CS - {noise}Hz', fontweight='bold')
    
    plt.ylim(-10, 117)
    
    plt.show()
    
    # Create the second plot: no_CS PCs with and without NO
    plt.figure(figsize=(8, 8))
    plt.rcParams.update({'font.size': 16})
    
    plt.errorbar(x, median_cr_no_cs, 
                 yerr=[median_cr_no_cs - q1_cr_no_cs, q3_cr_no_cs - median_cr_no_cs],
                 fmt='o-', 
                 label=f"without NO", 
                 capsize=5, 
                 markersize=3,
                 ecolor='black',
                 color='black')
    plt.errorbar(x, median_cr_no_cs_NO, 
                 yerr=[median_cr_no_cs_NO - q1_cr_no_cs_NO, q3_cr_no_cs_NO - median_cr_no_cs_NO],
                 fmt='o-', 
                 label=f"with NO", 
                 capsize=5, 
                 markersize=3,
                 ecolor=with_NO_color,
                 color=with_NO_color)
    
    plt.xlabel('Trial set [#]')
    plt.ylabel('CR [%]')
    plt.xticks([1, 5, 10, 15, 20, 25, 30])
    plt.title(f'Conditioned Response: DCN-noise - {noise}Hz',fontweight='bold' )
    
    plt.ylim(-10, 117)

    plt.show()